# Deterministic Agentic Query Decomposition with Hop-Limit Termination

**Problem Statement:** Naive multi-hop RAG agents enter infinite reformulation loops when corpora lack the requested evidence, burning tokens and timing out. Single-shot retrieval fails when answers require bridging two distinct policy documents.

**Success Metric:** Multi-hop Recall@10 >= 0.85 with <= 3.0 average hops and 0% infinite loops



In [ ]:
# Reproducibility & Structured Metrics Helper (100% Interactive Colab — No Tunnels/Servers)
import json, platform, sys, subprocess

def log_lab_environment(extra=None):
    gpu = "CPU only"
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
            stderr=subprocess.DEVNULL,
            text=True,
        ).strip()
        if out:
            gpu = out.splitlines()[0]
    except Exception:
        pass
    env_info = {
        "pythonVersion": platform.python_version(),
        "hardware": f"{gpu} ({platform.machine()})",
        "runtime": "Google Colab" if "google.colab" in sys.modules else platform.platform(),
    }
    if isinstance(extra, dict):
        env_info.update(extra)
    print("[LAB_ENV_JSON]", json.dumps(env_info))
    return env_info

def log_lab_result(metrics: dict):
    """Print structured metrics so the Lab workbench can import only compact numbers."""
    print("[LAB_RESULT_JSON]", json.dumps(metrics))

log_lab_environment()


## 1. Environment & Dependencies
Install required packages when running in Google Colab or a fresh Jupyter environment.

In [ ]:
import importlib.util, subprocess, sys
REQUIRED = ["numpy","pydantic","rank-bm25"]
missing = [p for p in REQUIRED if importlib.util.find_spec(p.replace("-", "_").split("==")[0]) is None]
if missing:
    print("Installing:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All required packages already installed:", REQUIRED)


## 2. Project Modules
The following cells sync the modular package files (`src/models.py`, `src/engine.py`, `src/__init__.py`) so this notebook runs identically both inside the repository and standalone in Google Colab.

### `src/models.py`

In [ ]:
from pathlib import Path
Path("src").mkdir(parents=True, exist_ok=True)
Path("src/__init__.py").touch(exist_ok=True)
Path("src/models.py").write_text("from pydantic import BaseModel, Field\nfrom typing import List, Optional\n\nclass Document(BaseModel):\n    doc_id: str\n    text: str\n    metadata: dict = Field(default_factory=dict)\n\nclass AgentState(BaseModel):\n    query: str\n    original_query: str\n    hop_count: int = 0\n    raw_memory: List[Document] = Field(default_factory=list)\n    compressed_history: str = \"\"\n    is_sufficient: bool = False\n\nclass SearchResult(BaseModel):\n    query: str\n    results: List[Document]\n", encoding="utf-8")
print("Synced module:", "src/models.py")


### `src/engine.py`

In [ ]:
from pathlib import Path
Path("src").mkdir(parents=True, exist_ok=True)
Path("src/__init__.py").touch(exist_ok=True)
Path("src/engine.py").write_text("import re\nimport numpy as np\nfrom typing import List\nfrom rank_bm25 import BM25Okapi\nfrom src.models import Document, AgentState\n\n\ndef _tokenize(text: str) -> List[str]:\n    return re.findall(r\"\\w+\", text.lower())\n\n\nclass MockRetriever:\n    def __init__(self, corpus: List[Document]):\n        self.corpus = corpus\n        tokenized_corpus = [_tokenize(doc.text) for doc in corpus]\n        self.bm25 = BM25Okapi(tokenized_corpus)\n\n    def search(self, query: str, top_k: int = 3) -> List[Document]:\n        tokenized_query = _tokenize(query)\n        if not tokenized_query:\n            return []\n        doc_scores = self.bm25.get_scores(tokenized_query)\n        top_indices = np.argsort(doc_scores)[::-1][:top_k]\n        return [self.corpus[i] for i in top_indices if doc_scores[i] > 0]\n\n\nclass Engine:\n    \"\"\"\n    Orchestrates the multi-hop process.\n    \"\"\"\n    def __init__(self, corpus: List[Document], max_hops: int = 3):\n        self.retriever = MockRetriever(corpus)\n        self.max_hops = max_hops\n\n    def _validator(self, state: AgentState) -> bool:\n        # Simulate LLM logic: check if raw_memory contains answer\n        # In a real impl, this would be an LLM call.\n        # Here, we look for keywords indicating \"complete\"\n        full_text = \" \".join([d.text for d in state.raw_memory]).lower()\n        required = [\"policy\", \"procedure\", \"limit\"]\n        return all(req in full_text for req in required)\n\n    def _summarize(self, state: AgentState) -> str:\n        # Mock summarization\n        learned = [d.text[:50] for d in state.raw_memory]\n        return f\"Previously explored: {'; '.join(learned)}\"\n\n    def _reformulate_query(self, state: AgentState) -> str:\n        # Simple heuristic: use original query on first hop, append context thereafter\n        if state.hop_count <= 1:\n            return state.original_query\n        return f\"{state.original_query} context: {state.compressed_history[:20]}\"\n\n    def run(self, query: str) -> AgentState:\n        state = AgentState(query=query, original_query=query)\n\n        for hop in range(self.max_hops):\n            state.hop_count = hop + 1\n\n            # 1. Reformulate\n            current_query = self._reformulate_query(state)\n\n            # 2. Retrieve\n            new_docs = self.retriever.search(current_query)\n            state.raw_memory.extend(new_docs)\n\n            # 3. Compress History\n            state.compressed_history = self._summarize(state)\n\n            # 4. Check Sufficiency\n            if self._validator(state):\n                state.is_sufficient = True\n                break\n\n        return state\n", encoding="utf-8")
print("Synced module:", "src/engine.py")


### `src/__init__.py`

In [ ]:
from pathlib import Path
Path("src").mkdir(parents=True, exist_ok=True)
Path("src/__init__.py").touch(exist_ok=True)
Path("src/__init__.py").write_text("# Mark src as a package\n", encoding="utf-8")
print("Synced module:", "src/__init__.py")


## 3. Run Benchmark & Evaluation (`main.py`)

In [ ]:
import sys
if "." not in sys.path:
    sys.path.insert(0, ".")

import json
from src.models import Document
from src.engine import Engine

def main():
    # Synthetic dataset
    corpus = [
        Document(doc_id="1", text="Company policy requires signed approvals."),
        Document(doc_id="2", text="The procedure for submittal involves a web form."),
        Document(doc_id="3", text="There is a hard limit of 5 days for review."),
        Document(doc_id="4", text="Irrelevant noise about coffee breaks.")
    ]

    engine = Engine(corpus, max_hops=3)
    
    # Run Benchmark
    query = "What is the policy procedure limit?"
    result = engine.run(query)

    # Calculate metrics
    metrics = {
        "success": result.is_sufficient,
        "hops_taken": result.hop_count,
        "raw_memory_size": len(result.raw_memory),
        "recall_met": result.is_sufficient # Simplified proxy
    }
    
    print(f"Orchestration complete. Hops: {result.hop_count}, Sufficient: {result.is_sufficient}")
    print("[LAB_RESULT_JSON]", json.dumps(metrics))

if __name__ == "__main__":
    main()